# EquiCave on Colab: one training run, resumable

This notebook does one thing: train **one arm** of the network on a Colab GPU and leave the result on Drive.
For the full tour of the pipeline use `notebooks/equicave_train.ipynb`; for the whole grid use your own
machine, because fifteen runs is sixty hours.

**Before anything else, check the GPU.** A run at the shipped configuration peaked at **22.5 GiB**, measured.

| GPU | memory | verdict |
|---|---|---|
| T4 (free) | 16 GB | does not fit. Smoke test only, with the overrides in the last section |
| L4 | 24 GB | fits with ~1.5 GiB spare |
| A100 40 GB | 40 GB | fits; this is the configuration the project measures |

**Hours.** About **4 h** for one run (17-20 epochs at ~13.5 min, measured on an A100 that was also sharing the
card, so a clean one should be faster). Plus **5 min** of setup if the feature cache is on Drive, or **2-3 h**
if it has to download structures and featurise them from scratch. Checkpointing is per epoch, so a dropped
session costs the current epoch, not the run.


In [ ]:
# 1. GPU and memory. Stop here if this says the card is too small for the configuration you intend.
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv


In [ ]:
# 2. Drive holds the cache and the runs, so a disconnect loses nothing.
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
D = pathlib.Path('/content/drive/MyDrive/equicave')
(D / 'runs').mkdir(parents=True, exist_ok=True)
os.environ['D'] = str(D)
print(sorted(p.name for p in D.iterdir()))


## Checkout and dependencies

The repository is private, so use a token (Settings -> Developer settings -> Personal access tokens, scope
`repo`) or upload a tarball of the checkout to Drive and extract it instead.


In [ ]:
TOKEN = ''  # paste a GitHub token, or leave empty and extract a tarball from Drive instead

import subprocess, sys, pathlib
REPO = pathlib.Path('/content/equicave')
if not REPO.exists():
    if TOKEN:
        url = f'https://{TOKEN}@github.com/Ruskinkot1/equicave.git'
        subprocess.run(['git', 'clone', '--depth', '1', '-b', 'claude/dazzling-fermat-x3au2t', url, str(REPO)], check=True)
    else:
        subprocess.run(['tar', '-xzf', str(D / 'equicave.tar.gz'), '-C', '/content'], check=True)
%cd /content/equicave
!pip -q install -e '.[train]' 2>&1 | tail -2


## The feature cache: copy it, do not rebuild it

Featurising 1367 structures takes over an hour and downloading them takes longer. The cache is npz files keyed
by PDB id plus a `.featurisation.json` signature, so copying `data/cache/net` (~1.2 GB) and
`data/processed/manifest.csv` from your machine to Drive turns the setup into a copy. The signature makes a
mismatched cache **refuse** rather than silently measure the wrong featurisation, and a complete cache no
longer loads ESM-2 at all (2.5 GB saved).

If the cache is not on Drive, the next cell builds it: expect two to three hours, and keep the session alive.


In [ ]:
CACHE = D / 'cache/net'            # copied from your machine, or built by the cell below
MANIFEST = D / 'manifest.csv'      # data/processed/manifest.csv from your checkout

import os
os.environ.update(CACHE=str(CACHE), MANIFEST=str(MANIFEST))
n = len(list(CACHE.glob('*.npz'))) if CACHE.exists() else 0
print(f'{n} structures cached at {CACHE}')
if n == 0:
    print('nothing cached: the run below will download structures and featurise them (2-3 h)')


## The run

`epochs=0` builds the cache and trains nothing, which is the cheap way to find out whether the data path
works before spending four hours. Then the real command. Re-running it after a disconnect **resumes**: the
checkpoint carries the weights, the EMA shadow, the optimiser, the schedule, the RNG stream and the
early-stopping counters.


In [ ]:
# dry run: cache only, no training
!PYTHONPATH=src:. python -m training pockets-net \
    --config training/configs/pockets_net.yaml --device cuda --out $D/runs \
    --set optim.epochs=0 data.cache_dir=$CACHE data.manifest=$MANIFEST


In [ ]:
# the run: `full`, fold 0, seed 0. About four hours. Safe to re-run after a disconnect.
!PYTHONPATH=src:. python -m training pockets-net \
    --config training/configs/pockets_net.yaml --device cuda --out $D/runs \
    --set ablation=full split.val_fold=0 optim.seed=0 tag=full \
          data.cache_dir=$CACHE data.manifest=$MANIFEST


## What came out


In [ ]:
import json, pathlib
run = sorted((D / 'runs').glob('full_fold0_seed*'))[-1]
m = json.load(open(run / 'metrics.json'))
print(run.name)
print('site top-1      ', round(m['net_sites']['top1'], 3))
print('site top-(N+2)  ', round(m['net_sites']['topN2'], 3))
print('centre error med', round(m['net_center_error_median'], 2), 'A')
print('occ AP / res AP ', round(m['occ_ap'], 3), '/', round(m['res_ap'], 3))
print('val_score       ', round(m['val_score'], 3))
print()
print('by pocket class (top-1, ceiling, n):')
for k, v in sorted(m.get('net_sites_by_class', {}).items(), key=lambda kv: kv[1].get('top1', 9)):
    if 'top1' in v:
        print(f"  {k:16s} {v['top1']:.3f}  ceiling {v['ceiling']:.3f}  n={v['n']}")


## A 16 GB T4: smoke test only

Every override below changes the configuration, so such a run shows the pipeline works and says **nothing**
about accuracy. Its numbers do not go in a table. `use_tensors=false` is the largest single saving — the
degree-2 edge messages are `[E, F, 3, 3]`, about 53 MiB per tensor at 24 k edges and 128 channels, several per
layer, five layers, three recycling passes, each kept by autograd — and it costs −0.006 top-1, which is
nothing. The rest is capacity, which is not free.


In [ ]:
!PYTHONPATH=src:. python -m training pockets-net \
    --config training/configs/pockets_net.yaml --device cuda --out $D/runs \
    --set ablation=full tag=smoke optim.epochs=2 stages.warmup_epochs=1 \
          model.use_tensors=false model.dim=96 data.n_probe=384 optim.need_gb=12 \
          data.cache_dir=$CACHE data.manifest=$MANIFEST
